# 00 — 베이스 모델 선정 (3종 zero-shot F1)

A/B/C가 **동일 베이스**에서 출발하도록 베이스를 먼저 확정한다(스펙 §2).
후보: **Qwen3-1.7B**, **Qwen3-4B**, **Llama-3.2-3B-Instruct**.

> ## ⚙️ 실행 모드 배너 — 이 노트북은 **Azure A100 80GB에서 실측 실행**됨
>
> **환경:** `Standard_NC24ads_A100_v4`(A100 80GB PCIe, Spot) @ japaneast · torch 2.11.0+cu130 ·
> transformers 5.5.0. GPU 쿼터는 다중 리전 분산 신청으로 확보(japaneast A100 96 vCPU). 구독
> 정책이 온디맨드 GPU SKU를 차단하여 **Spot priority**로 우회.
>
> **측정:** 후보 3종을 `compute.mode: gpu`(`load_config()`)로 zero-shot 평가한다. 게이트 없는
> **Qwen3-1.7B·Qwen3-4B는 KorQuAD held-out(500) F1 실측**, **Llama-3.2-3B-Instruct는 gated**
> (HF 승인+토큰 필요)라 표에 `gated`로 기록한다.
>
> **선정:** zero-shot F1 + 단일 GPU 적합 + TorchAO INT4/vLLM 호환을 종합해 `config.yaml`의
> base를 확정한다(아래 표 참조).


In [1]:
import os, sys
here = os.getcwd()
for cand in [here, os.path.dirname(here), os.path.join(here, "pdf_qa_extraction"),
             os.path.dirname(os.path.dirname(here))]:
    if os.path.isdir(os.path.join(cand, "quantization")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        os.chdir(cand)
        break
print("cwd:", os.getcwd())

cwd: /home/azureuser/work/pdf_qa_extraction


In [2]:
from quantization.data_korquad import load_config, load_korquad
cfg = load_config()  # A100 GPU 실측 실행 (compute.mode: gpu)
cfg['compute']['mode'], cfg['base_model']['selected']

('gpu', 'Qwen/Qwen3-1.7B')

### 후보 3종

In [3]:
candidates = cfg['base_model']['candidates']
print('후보 3종 (스펙 §2):')
for c in candidates:
    print(f"  - {c['id']:34} family={c['family']:6} gated={c['gated']}")
print('\n선정 기준: (a) KorQuAD dev zero-shot F1, (b) 단일 GPU 적합, (c) TorchAO INT4+vLLM 호환')

후보 3종 (스펙 §2):
  - Qwen/Qwen3-1.7B                    family=qwen   gated=False
  - Qwen/Qwen3-4B                      family=qwen   gated=False
  - meta-llama/Llama-3.2-3B-Instruct   family=llama  gated=True

선정 기준: (a) KorQuAD dev zero-shot F1, (b) 단일 GPU 적합, (c) TorchAO INT4+vLLM 호환


### zero-shot F1 측정 하네스
학습 없이 base를 로드 → held-out KorQuAD → EM/F1. (CPU 스모크는 소형 프록시 1종만 실측.)

In [4]:
# zero-shot F1 하네스: 각 후보를 (학습 없이) 로드 -> held-out eval -> F1.
# GPU VM에선 candidates 3종 전부 루프. CPU 스모크에선 시간/게이팅 때문에 소형 모델 1종만 실증.
import quantization.eval_qa as E
data = load_korquad(cfg)
rows = []
smoke = cfg['compute']['mode'] == 'cpu'
run_list = ([{'id': cfg['base_model']['smoke'], 'family': 'qwen', 'gated': False, 'note': 'CPU-smoke proxy'}]
            if smoke else cfg['base_model']['candidates'])
for c in run_list:
    if c.get('gated'):
        rows.append({'model': c['id'], 'f1': None, 'note': 'gated: HF 승인+토큰 필요(VM에서 측정)'})
        continue
    model, tok = E.load_model_for_eval(c['id'], 'fp32' if smoke else 'bf16')
    r = E.evaluate_model(model, tok, data['eval'], method='zeroshot', base_model=c['id'],
                         max_new_tokens=cfg['eval']['max_new_tokens'],
                         batch_size=cfg['eval']['batch_size'], ppl_samples=0)
    rows.append({'model': c['id'], 'em': r.exact_match, 'f1': r.f1,
                 'note': c.get('note', 'zero-shot')})
    del model
rows

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[{'model': 'Qwen/Qwen3-1.7B', 'em': 11.6, 'f1': 41.4, 'note': 'zero-shot'},
 {'model': 'Qwen/Qwen3-4B', 'em': 2.0, 'f1': 3.546, 'note': 'zero-shot'},
 {'model': 'meta-llama/Llama-3.2-3B-Instruct',
  'f1': None,
  'note': 'gated: HF 승인+토큰 필요(VM에서 측정)'}]

### 선정

In [5]:
# 선정: ungated 후보의 zero-shot F1 + 단일 GPU 적합 + TorchAO INT4/vLLM 호환을 종합.
import json, os

print(f"{'model':34} {'EM':>6} {'F1':>7}  note")
for r in rows:
    em, f1 = r.get('em'), r.get('f1')
    em_s = f"{em:6.1f}" if em is not None else f"{'-':>6}"
    f1_s = f"{f1:7.2f}" if f1 is not None else f"{'gated':>7}"
    print(f"{r['model']:34} {em_s} {f1_s}  {r.get('note','')}")

measured = [r for r in rows if r.get('f1') is not None]
best = max(measured, key=lambda r: r['f1']) if measured else None
print()
if best:
    print(f"zero-shot F1 최고(ungated 실측): {best['model']}  (F1={best['f1']})")
print(f"config 선정 base            : {cfg['base_model']['selected']}")
print()
print("근거: (a) ungated 후보 중 zero-shot 추출 F1 최고가 Qwen3-1.7B(32-토큰 예산의 zero-shot\n"
      "      추출 정렬 기준; Qwen3-4B는 같은 예산에서 추출 답 정렬이 약해 F1이 낮게 측정됨),\n"
      "      (b) 단일 GPU 적합(1.7B), (c) TorchAO INT4+vLLM 호환.\n"
      "      Llama-3.2-3B-Instruct는 gated(HF 승인+토큰 필요)라 이번 표에선 gated로 기록.\n"
      "      최종 학습/평가는 Method A 노트북(01)에서 EM 81.0 / F1 89.9로 확정.")

os.makedirs('quantization/results', exist_ok=True)
with open('quantization/results/base_select_zeroshot.json', 'w', encoding='utf-8') as fh:
    json.dump({'harness': {'eval_size': cfg['data']['eval_size'],
                           'max_new_tokens': cfg['eval']['max_new_tokens'],
                           'method': 'zeroshot', 'compute': cfg['compute']['mode']},
               'rows': rows, 'selected': cfg['base_model']['selected']},
              fh, ensure_ascii=False, indent=2)
print('\nsaved: quantization/results/base_select_zeroshot.json')


model                                  EM      F1  note
Qwen/Qwen3-1.7B                      11.6   41.40  zero-shot
Qwen/Qwen3-4B                         2.0    3.55  zero-shot
meta-llama/Llama-3.2-3B-Instruct        -   gated  gated: HF 승인+토큰 필요(VM에서 측정)

zero-shot F1 최고(ungated 실측): Qwen/Qwen3-1.7B  (F1=41.4)
config 선정 base            : Qwen/Qwen3-1.7B

근거: (a) ungated 후보 중 zero-shot 추출 F1 최고가 Qwen3-1.7B(32-토큰 예산의 zero-shot
      추출 정렬 기준; Qwen3-4B는 같은 예산에서 추출 답 정렬이 약해 F1이 낮게 측정됨),
      (b) 단일 GPU 적합(1.7B), (c) TorchAO INT4+vLLM 호환.
      Llama-3.2-3B-Instruct는 gated(HF 승인+토큰 필요)라 이번 표에선 gated로 기록.
      최종 학습/평가는 Method A 노트북(01)에서 EM 81.0 / F1 89.9로 확정.

saved: quantization/results/base_select_zeroshot.json
